<a href="https://colab.research.google.com/github/amilafr/algo-python-pro2/blob/main/Panda3d_lengkap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# game.py

In [ ]:
# connect base scene class
from direct.showbase.ShowBase import ShowBase
from mapmanager import *
from hero import *

class Game(ShowBase):
    def __init__(self):
        ShowBase.__init__(self)
        self.land = Mapmanager()
        # self.land.loadLand("land.txt")
        # self.land.loadLand("land2.txt")
        x, y = self.land.loadLand("land.txt")
        self.hero = Hero((x//2, y//2, 2), self.land)
        base.camLens.setFov(90)

game = Game()
game.run()

# mapmanager.py

In [ ]:
import pickle

class Mapmanager():

    """ Managing map """
    def __init__(self):
        #   Load a block model, set a texture, create a property for the color, model and texture
        self.model = 'block.egg'
        self.texture = 'block.png'
        # self.color = (0.2, 0.2, 0.35, 1) # rgba
        self.colors = [
            (0.5, 0.3, 0.0, 1),
            (0.2, 0.2, 0.3, 1),
            (0.5, 0.5, 0.2, 1),
            (0.0, 0.6, 0.0, 1)
        ]

        # create nodes
        self.startNew()
        # add block
        # self.addBlock((0, 10, 0))

    def startNew(self):
        self.land = render.attachNewNode("Land")

    def addBlock(self, position):
        self.block = loader.loadModel(self.model)
        self.block.setTexture(loader.loadTexture(self.texture))
        self.block.setPos(position)
        # self.block.setColor(self.color)
        self.block.reparentTo(self.land)

        self.color = self.getColor(position[2])
        self.block.setColor(self.color)

        self.block.setTag("at", str(position))

    # hapus block
    def delBlock(self, position):
        blocks = self.findBlocks(position)
        for block in blocks:
            block.removeNode()

    # nambah block
    def buildBlock(self, pos):
        x,y,z = pos
        new = self.findHighestEmpty(pos)
        if new[2] <= z + 1:
            self.addBlock(new)

    def delBlockFrom(self, position):
        x,y,z = position
        pos = x, y, z-1
        blocks = self.findBlocks(pos)
        for block in blocks:
            block.removeNode()

    def getColor(self, z):
        if z < len(self.colors):
            return self.colors[z]
        else:
            return self.colors[-1]

    def loadLand(self, filename):
        self.clear()
        with open(filename) as file:
            y = 0
            for line in file:
                x = 0
                line = line.split(' ')
                for z in line:
                    for z0 in range(int(z)+1):
                        block = self.addBlock((x, y, z0))
                    x += 1
                y += 1

        return x, y

    def clear(self):
        self.land.removeNode() # remove yg old
        self.startNew()

    def isEmpty(self, pos): # kosong nggak
        blocks = self.findBlocks(pos) # nyari
        if blocks: # kalo ada block
            return False
        else:
            return True

    def findBlocks(self, pos): # nemuin blocks
        return self.land.findAllMatches("=at=" + str(pos))

    def findHighestEmpty(self, pos):
        x, y, z = pos
        z = 1
        while not self.isEmpty((x, y, z)):
            z += 1
        return (x, y, z)

    def saveMap(self): # nyimpen
        blocks = self.land.getChildren()
        # simpen map --> ke dalam file my_map.dat
        with open('my_map.dat', 'wb') as f: # write file
            pickle.dump(len(blocks), f)

            for block in blocks:
                x, y, z = block.getPos()
                pos = (int(x), int(y), int(z))
                pickle.dump(pos, f)

    def loadMap(self): # load map
        self.clear()
        # buka file my_map
        with open('my_map.dat', 'rb') as f:
            # load
            length = pickle.load(f)
            for i in range(length):
                pos = pickle.load(f)
                self.addBlock(pos)

# hero.py

In [ ]:
class Hero():
    def __init__(self, pos, land):
        self.land = land
        self.hero = loader.loadModel('smiley')
        self.hero.setColor(1, 0.5, 0)
        self.hero.setScale(0.3)
        self.hero.setPos(pos)
        self.hero.reparentTo(render)

        self.mode = True

        self.cameraBind()
        self.accept_events()

    # player mode
    def cameraBind(self):
        base.disableMouse()
        base.camera.setH(180) # rotate
        base.camera.reparentTo(self.hero)
        base.camera.setPos(0, 0, 1.5)
        self.cameraOn = True

    # spectator mode
    def cameraUp(self):
        pos = self.hero.getPos()
        base.mouseInterfaceNode.setPos(-pos[0],-pos[1], -pos[2] - 3)
        base.camera.reparentTo(render)
        base.enableMouse()
        self.cameraOn = False

    def changeView(self):
        if self.cameraOn:
            self.cameraUp()
        else:
            self.cameraBind()

    def turn_left(self):
        self.hero.setH((self.hero.getH() + 5) % 360)

    def turn_right(self):
        self.hero.setH((self.hero.getH() - 5) % 360)

    def just_move(self, angle):
        pos = self.look_at(angle)
        self.hero.setPos(pos)

    def try_move(self, angle):
        pos = self.look_at(angle)
        if self.land.isEmpty(angle):
            pos = self.land.findHighestEmpty(pos)
            self.hero.setPos(pos)
        else:
            pos = pos[0], pos[1], pos[2] + 1
            if self.land.isEmpty(pos):
                self.hero.setPos(pos)

    def move_to(self, angle):
        if self.mode:
            self.just_move(angle)
        else:
            self.try_move(angle)

    def look_at(self, angle):
        from_x = round(self.hero.getX())
        from_y = round(self.hero.getY())
        from_z = round(self.hero.getZ())

        dx, dy = self.check_dir(angle)

        return from_x + dx, from_y + dy, from_z

    def check_dir(self, angle):
        if angle >= 0 and angle <= 20:
            return 0, -1
        elif angle > 20 and angle <= 65:
            return 1, -1
        elif angle > 65 and angle <= 110:
            return 1, 0
        elif angle > 110 and angle <= 155:
            return 1, 1
        elif angle > 155 and angle <= 200:
            return 0, 1
        elif angle > 200 and angle <= 245:
            return -1, 1
        elif angle > 245 and angle <= 290:
            return -1, 0
        elif angle > 290 and angle <= 335:
            return -1, -1
        else:
            return 0, -1

    def back(self):
        angle = (self.hero.getH() + 180) % 360
        self.move_to(angle)

    def forward(self):
        angle = (self.hero.getH()) % 360
        self.move_to(angle)

    def left(self):
        angle = (self.hero.getH() + 90) % 360
        self.move_to(angle)

    def right(self):
        angle = (self.hero.getH() + 270) % 360
        self.move_to(angle)

    def up(self):
        self.hero.setZ(self.hero.getZ() + 1)

    def down(self):
        self.hero.setZ(self.hero.getZ() - 1)

    def changeMode(self):
        if self.mode == True:
            self.mode = False
        else:
            self.mode = True

    def build(self):
        angle = self.hero.getH() % 360
        pos = self.look_at(angle)

        if self.mode:
            self.land.addBlock(pos)
        else:
            self.land.buildBlock(pos)

    def destroy(self):
        angle = self.hero.getH() % 360
        pos = self.look_at(angle)

        if self.mode:
            self.land.delBlock(pos)
        else:
            self.land.delBlockFrom(pos)

    # events
    def accept_events(self):
        base.accept('c', self.changeView)

        # turn left
        base.accept('n', self.turn_left)
        base.accept('n' + '-repeat', self.turn_left)

        # turn right
        base.accept('m', self.turn_right)
        base.accept('m' + '-repeat', self.turn_right)

        # forward
        base.accept('w', self.forward)
        base.accept('w' + '-repeat', self.forward)

        # back
        base.accept('s', self.back)
        base.accept('s' + '-repeat', self.back)

        # right
        base.accept('d', self.right)
        base.accept('d' + '-repeat', self.right)

        # left
        base.accept('a', self.left)
        base.accept('a' + '-repeat', self.left)

        # upward
        base.accept('e', self.up)
        base.accept('e' + '-repeat', self.up)

        # downward
        base.accept('q', self.down)
        base.accept('q' + '-repeat', self.down)

        # change mode
        base.accept('z', self.changeMode)

        # build
        base.accept('b', self.build)

        # destroy
        base.accept('v', self.destroy)

        # savemap
        base.accept('k', self.land.saveMap)

        # loadmap
        base.accept('l', self.land.loadMap)
